# 05 — Metal inference on TestSet300
Locked metal model (Dev-only decision of GEAR_METAL_ONLY_ULTIMATE_V2): **MLP on ESM + ATOM + CHEM + MULTI2**,
ensemble = mean of the 5 official-fold models.

- Fold weights: `final_model/metal/mlp/*.pt` (Colab T4 rerun of the original notebook, `notebooks/07`; all saved
  predictions bit-identical to the original run). Nothing is trained; `run_version=` selects other weights.
- Thresholds: 0.5 and the Dev-OOF F1-optimal threshold of the same five models (backend convention).
- Outputs: `results/METAL_INFERENCE/ESM__Atomic__Chem__MULTI2/TestSet300/`.

In [ ]:
# ============================================================
# BOOTSTRAP — Colab: mount Drive, put pbind on the path
# ============================================================
import sys, subprocess
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass

# Folder of this repo (copy it to Drive once, or git clone it into /content).
REPO = Path("/content/drive/MyDrive/Protein-binding-Phi/protein-binding")
sys.path.insert(0, str(REPO / "src"))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements.txt")], check=True)

# Data root (Drive layout). Override with PBIND_BASE for a local mirror.
# import os; os.environ["PBIND_BASE"] = "/content/drive/MyDrive/Protein-binding-Phi"

from pbind.paths import resolve_paths, check_inputs
display(check_inputs(resolve_paths()))

In [ ]:
from pbind.pipelines import open_workspace
from pbind.inference import run_metal_inference, locked_config

ws = open_workspace("metal", attach_evo=False, attach_esmif=False)
locked_config(ws)

In [ ]:
R = run_metal_inference(ws, split="TestSet300")     # run_version=... for other weights
R["summary"]

## Metrics (global residue + protein-mean, both thresholds)

In [ ]:
display(R["metrics"][["threshold_rule", "threshold", "AUPRC_global_residue", "AUROC_global_residue",
                      "Precision_positive_global_residue", "Recall_positive_global_residue",
                      "F1_positive_global_residue", "MCC_global_residue",
                      "protein_mean_F1_baseline_convention", "protein_mean_MCC_baseline_convention"]])

## Per-protein and per-residue predictions

In [ ]:
display(R["proteins"].sort_values("max_prob", ascending=False).head(20))
display(R["residues"].head())